In [1]:
from datamodules.bert_datamodule import BertDataModule
import hydra
from models.bert_module import CustomBertModelModule
import pytorch_lightning as pl
from pytorch_lightning import loggers as pl_loggers
from aggregator import Aggregator
from client import Client
import torch.multiprocessing as mp
import torch
import copy
import uuid
import tqdm
from pytorch_lightning.utilities.parsing import AttributeDict

from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.nn.parameter import Parameter
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
from transformers import AdamW, get_linear_schedule_with_warmup, BertForSequenceClassification, BertTokenizer
from typing import Dict, List, Tuple

/home/aladin/projects/resilient_sfl/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Get dataloaders
data_config = AttributeDict({
            "glue_dataset": "sst2",
            "batch_size": 256,
            "num_workers": 1,
            "model_type": "bert-base-uncased"
            })

datamodule = BertDataModule(**data_config)
datamodule.prepare_data()
datamodule.setup()

train_dataloader = datamodule.train_dataloader()
val_dataloader = datamodule.val_dataloader()

Found cached dataset glue (/home/aladin/.cache/huggingface/datasets/glue/sst2/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)
100%|██████████| 3/3 [00:00<00:00, 647.34it/s]


In [3]:
# Load master model
model_path = "/home/aladin/projects/resilient_sfl/src/results/master/dff08f15-00f6-4fbb-8081-8483374c897c_master.pt"
master_model = torch.load(model_path)

# Create client
client = Client(id=0,
                model=CustomBertModelModule.from_pretrained("bert-base-uncased", num_labels=2),
                trainer=pl.Trainer(),
                train_data=train_dataloader,
                val_data=val_dataloader) 

# Assign master model to client
client.model.load_state_dict(master_model)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing CustomBertModelModule: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing CustomBertModelModule from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing CustomBertModelModule from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of CustomBertModelModule were not initialized from the model checkpoint at bert-base-uncased and are newly

<All keys matched successfully>

In [4]:
# Load dataloaders from BERT
class Manager():
    """
    Manager class for testing SFL. Handles model loading, data preprocessing, etc.
    """
    def __init__(self, 
                 name: str = "bert_manager",
                 model_type: str = "bert-base-uncased", 
                 batch_size: int = 256) -> None:
        self.name = name
        self.model_type = model_type
        self.batch_size = batch_size


    def load_model(self, num_labels: int) -> BertForSequenceClassification:
        """
        Loads the specified BERT model.
        """
        self.model = BertForSequenceClassification.from_pretrained(self.model_type, num_labels=num_labels)
        return self.model


    def tokenize(self, dataset) -> torch.utils.data.TensorDataset:
        """
        Tokenizes the dataset.
        """
        self.tokenizer = BertTokenizer.from_pretrained(self.model_type)
        encodings = self.tokenizer(dataset["sentence"], truncation=True, padding=True)
        labels = torch.tensor(dataset["label"], dtype=torch.long)
        input_ids = torch.tensor(encodings['input_ids'])
        attention_mask = torch.tensor(encodings['attention_mask'])
        dataset = torch.utils.data.TensorDataset(input_ids, attention_mask, labels)
        return dataset  


    def preprocess_dataset(self, 
                           glue_dataset: str = "sst2", 
                           truncate: int = None
                           ) -> Tuple[DataLoader, DataLoader]:
        """
        Preprocesses the dataset for SFL: tokenization, truncation, DataLoader
        """

        # Load dataset
        dataset = load_dataset("glue", glue_dataset)
        train_dataset, valid_dataset, test_dataset = dataset["train"], dataset["validation"], dataset["test"]

        # Truncate dataset
        if truncate is not None:
            train_dataset = train_dataset.select(range(truncate))
            valid_dataset = valid_dataset.select(range(truncate))
            test_dataset = test_dataset.select(range(truncate))
        else:
            pass

        # Tokenize dataset
        train_dataset = self.tokenize(train_dataset)
        valid_dataset = self.tokenize(valid_dataset)
        test_dataset = self.tokenize(test_dataset)

        # Create dataloaders
        train_dataloader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        valid_dataloader = DataLoader(valid_dataset, batch_size=self.batch_size, shuffle=True)
        test_dataloader = DataLoader(test_dataset, batch_size=self.batch_size, shuffle=True)

        return train_dataloader, valid_dataloader, test_dataloader


    def split_data(self, 
                   data: DataLoader, 
                   clients: int = 2) -> List[DataLoader]:
        """
        Splits the training and validation datasets into equally sized client portions.
        """
        dataset = data.dataset
        split_size = len(dataset) // clients
        remainder = len(dataset) % clients

        split_lengths = [split_size + 1 if i < remainder else split_size for i in range(clients)]
        splits = random_split(dataset, split_lengths)

        split_dataloaders = [DataLoader(split, batch_size=data.batch_size) for split in splits]

        return split_dataloaders


    def save_plots(self,
                   loss: List[float], 
                   accuracy: List[float],
                   title: str,
                   path= str) -> None:
        """
        Saves the loss and accuracy plots.
        """
        plt.figure(figsize=(10, 5))

        epochs = range(1, len(loss) + 1)

        # Plot the loss values
        plt.subplot(1, 2, 1)
        plt.plot(epochs, loss, 'r', label='Loss')
        plt.title('Training Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()

        # Plot the accuracy values
        plt.subplot(1, 2, 2)
        plt.plot(epochs, accuracy, 'b', label='Training Accuracy')
        plt.title('Training vs. Validation Accuracy')
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy')
        plt.legend()

        # Title
        plt.suptitle(title)

        # Save the plot
        plt.savefig(path)


    def save_metric(self, metric: List[float], path: str) -> None:
        """
        Saves the metric (e.g. loss or accuracy) to the specified path.
        """
        np.array(metric)
        np.save(path, metric)


    def load_metric(self, path: str) -> List[float]:
        """
        Loads the metric (e.g. loss or accuracy) from the specified path.
        """
        metric = np.load(path)
        return metric


manager = Manager(name="bert-manager", model_type="bert-base-uncased", batch_size=256)
t_dl, v_dl, te_dl = manager.preprocess_dataset(glue_dataset="sst2")

Found cached dataset glue (/home/aladin/.cache/huggingface/datasets/glue/sst2/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)
100%|██████████| 3/3 [00:00<00:00, 834.13it/s]


In [8]:
# Evaluate master model on TRAIN DATA
total_loss = 0
total_correct = 0
total_samples = 0

for step, batch in enumerate(tqdm(t_dl)):

    device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
    client.model.to(device)
    batch = tuple(t.to(device) for t in batch)
    inputs = {
        "input_ids": batch[0],
        "attention_mask": batch[1],
        "labels": batch[2]
    }

    with torch.no_grad():
        outputs = client.model(**inputs)

    eval_loss = outputs.loss
    logits = outputs.logits

    predictions = torch.argmax(logits, dim=1)
    true_labels = batch[2]

    total_correct += (predictions == true_labels).sum().item()
    total_samples += len(true_labels)
    total_loss += eval_loss.item()

# Compute average loss and accuracy
average_loss = total_loss / len(t_dl)
accuracy = total_correct / total_samples

print(f"Average Loss: {average_loss:.4f}")
print(f"Accuracy: {accuracy:.4f}")

100%|██████████| 264/264 [01:35<00:00,  2.76it/s]

Average Loss: 0.6926
Accuracy: 0.5320


In [7]:
# Evaluate master model on VAL DATA
total_loss = 0
total_correct = 0
total_samples = 0

for step, batch in enumerate(tqdm(v_dl)):

    device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
    client.model.to(device)
    batch = tuple(t.to(device) for t in batch)
    inputs = {
        "input_ids": batch[0],
        "attention_mask": batch[1],
        "labels": batch[2]
    }

    with torch.no_grad():
        outputs = client.model(**inputs)

    eval_loss = outputs.loss
    logits = outputs.logits

    predictions = torch.argmax(logits, dim=1)
    true_labels = batch[2]

    total_correct += (predictions == true_labels).sum().item()
    total_samples += len(true_labels)
    total_loss += eval_loss.item()

# Compute average loss and accuracy
average_loss = total_loss / len(v_dl)
accuracy = total_correct / total_samples

print(f"Average Loss: {average_loss:.4f}")
print(f"Accuracy: {accuracy:.4f}")

100%|██████████| 4/4 [00:01<00:00,  3.66it/s]

Average Loss: 0.7095
Accuracy: 0.4725
